# Intensity Distributions
Plot the normalized distributions of intensity and log-intensity for fully refracted paths and boundary reflecting paths

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
from dotenv import find_dotenv, load_dotenv

import colorsys

load_dotenv(find_dotenv())

%config InlineBackend.figure_format = 'retina'

### Poster presentation settings

In [ ]:
import matplotlib.font_manager as fm

# Add the font file
font_path = '/Users/john/Library/Fonts/RedHatDisplay-Medium.ttf'  # Update this path
fm.fontManager.addfont(font_path)

# Use the font
plt.rcParams['font.family'] = 'Red Hat Display'

In [ ]:
# Define your colors
whoi_colors = ['#00B7BD', '#EE5340', '#0069b1', '#041e42', '#00a9e0']

In [ ]:
def lighten_color(hex_color, saturation_scale=0.5, lightness_scale=1.5):
    """
    Lighten a hex color similar to seaborn's tab20 palette
    
    Args:
        hex_color: Hex color string (e.g., '#1f77b4')
        saturation_scale: Multiply saturation by this (default 0.5)
        lightness_scale: Multiply lightness by this (default 1.5)
    """
    # Remove '#' if present
    hex_color = hex_color.lstrip('#')
    
    # Convert hex to RGB (0-1 range)
    r, g, b = tuple(int(hex_color[i:i+2], 16) / 255.0 for i in (0, 2, 4))
    
    # Convert RGB to HLS
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    
    # Adjust saturation and lightness
    s_new = min(1.0, s * saturation_scale)
    l_new = min(1.0, l * lightness_scale)
    
    # Convert back to RGB
    r_new, g_new, b_new = colorsys.hls_to_rgb(h, l_new, s_new)
    
    # Convert to hex
    return '#{:02x}{:02x}{:02x}'.format(
        int(r_new * 255),
        int(g_new * 255),
        int(b_new * 255)
    )

In [ ]:
colors = [
    whoi_colors[0],
    lighten_color(whoi_colors[0], lightness_scale=2),
    whoi_colors[1],
    lighten_color(whoi_colors[1], lightness_scale=1.4, saturation_scale=1)
]

In [ ]:
colors

In [ ]:
plt.plot(np.arange(0,10), lw=2, c=colors[0])
plt.plot(np.arange(2,12), lw=2, c=colors[1])
plt.plot(np.arange(4,14), lw=2, c=colors[2])
plt.plot(np.arange(6,16), lw=2, c=colors[3])

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/wavefront_values.pkl"

with open(fn, "rb") as f:
    wavefront_values = pickle.load(f)

## Intensity Distributions

In [ ]:
ducted_peaks = ["+7", "-7", "+6", "-6"]

boundary_peaks = [
    "+7b",
    "-7b",
    "+8b",
    "-8b",
    "+9b",
    "-9b",
    "+10b",
    "-10b",
    "+11b",
    "-11b",
    "+12b",
    "-12b",
    "+13b",
    "-13b",
    "+14b",
    "-14b",
    "+15b",
    "-15b",
]

I = {}
lI = {}
Ibar = {}
lIbar = {}
sigma_l = {}
I_norm = {}
lI_norm = {}
SI = {}

for ray_id in wavefront_values.keys():
    I[ray_id] = np.abs(wavefront_values[ray_id]) ** 2
    lI[ray_id] = np.log(I[ray_id])

    Ibar[ray_id] = np.mean(I[ray_id])
    lIbar[ray_id] = np.mean(lI[ray_id])
    sigma_l[ray_id] = np.std(lI[ray_id])

    I_norm[ray_id] = (I[ray_id] / Ibar[ray_id]).flatten()
    lI_norm[ray_id] = ((lI[ray_id] - lIbar[ray_id]) / sigma_l[ray_id]).flatten()

    SI[ray_id] = (np.mean(I[ray_id] ** 2) - np.mean(I[ray_id]) ** 2) / (
        np.mean(I[ray_id]) ** 2
    )

In [ ]:
plt.rcParams.update({"font.size": 8})
plt.rcParams.update({"text.usetex": False})

fig, axes = plt.subplots(2, 4, figsize=(6.68, 3.5))
axes = axes.flatten()

for k, ray_id in enumerate(ducted_peaks):
    # Intensity Distribution
    plt.sca(axes[k])

    _ = plt.hist(
        I_norm[ray_id],
        bins=np.linspace(0, 10, 40),
        alpha=0.3,
        density=True,
        color='gray'
    )

    plt.scatter(
        y=3e-3 * np.random.uniform(0.7, 1.3, 644),
        x=I_norm[ray_id].flatten(),
        lw=0,
        alpha=0.3,
        s=1,
        c="k",
    )

    x_plot = np.linspace(0, 10, 1000)
    # exponential distribution
    plt.plot(x_plot, np.exp(-x_plot), c=whoi_colors[1])

    # log-normal distribution
    sigma = 1
    lognormal_f = (
        1
        / (x_plot * np.sqrt(2 * np.pi) * sigma)
        * np.exp(-1 * (np.log(x_plot) ** 2) / (2 * (sigma**2)))
    )
    plt.plot(x_plot, lognormal_f, c=whoi_colors[1], linestyle="--")

    plt.yscale("log")
    plt.ylim([1e-3, 1e0])
    # plt.title(ray_id, loc='left', fontsize=10, pad=-3)

    plt.text(5.6, 2.5e-1, f"Ray ID: {ray_id}\nSI = {SI[ray_id]:0.2f}")

    plt.xlabel("$ I / \\langle I \\rangle$")
    plt.xlim([0, 10])
    plt.xticks(np.arange(0, 10, 3))

    # Log-Intensity Distribution
    plt.sca(axes[k + 4])
    _ = plt.hist(
        lI_norm[ray_id],
        bins=np.linspace(-6, 3, 40),
        alpha=0.3,
        density=True,
        color='gray'
    )

    plt.scatter(
        y=3e-3 * np.random.uniform(0.7, 1.3, 644),
        x=lI_norm[ray_id].flatten(),
        lw=0,
        alpha=0.3,
        s=1,
        c="k",
    )

    x_plot = np.linspace(-6, 4, 1000)
    plt.plot(x_plot, np.exp(x_plot - np.exp(x_plot)), linestyle='-', color=whoi_colors[1])
    sigma = 1
    f_norm = 1 / (sigma * np.sqrt(2 * np.pi)) * np.exp(-0.5 * (x_plot / sigma) ** 2)
    plt.plot(x_plot, f_norm, linestyle='--', color=whoi_colors[1])

    plt.yscale("log")
    plt.ylim([1e-3, 1e0])
    plt.xlim([-6, 4])
    plt.xticks(np.arange(-6, 4, 3))
    plt.xlabel(
        "$(\\mathrm{ln \\ I} - \\langle \\mathrm{ln \\ I} \\rangle) / \\sigma_{\\mathrm{ln \\ I}}$"
    )
    # plt.title(ray_id, loc='left', fontsize=10, pad=-3)

    plt.text(-5.2, 2.5e-1, f"Ray ID: {ray_id}\nSI = {SI[ray_id]:0.2f}")


# axes level formatting
for k in range(8):
    if k % 4 == 0:
        axes[k].set_ylabel("Probability Density")
    else:
        axes[k].set_yticklabels("")

plt.tight_layout(pad=0, h_pad=2)

#fig.savefig("../../figures/Ducted_Intensity_Distribution.pdf", bbox_inches="tight")

In [ ]:
plt.rcParams.update({"font.size": 8})
plt.rcParams.update({"text.usetex": True})

fig, axes = plt.subplots(6, 3, figsize=(6.68, 7))
axes = axes.flatten()

for k, ray_id in enumerate(boundary_peaks):
    # Intensity Distribution
    plt.sca(axes[k])

    _ = plt.hist(
        I_norm[ray_id],
        bins=np.linspace(0, 10, 40),
        alpha=0.3,
        density=True,
        color="gray",
    )

    plt.scatter(
        y=3e-3 * np.random.uniform(0.7, 1.3, 644),
        x=I_norm[ray_id].flatten(),
        lw=0,
        alpha=0.3,
        s=1,
        c="k",
    )

    x_plot = np.linspace(0, 8, 1000)
    plt.plot(x_plot, np.exp(-x_plot), c="k")

    plt.yscale("log")
    plt.ylim([1e-3, 1e0])
    # plt.title(ray_id, loc='left', fontsize=10, pad=-3)

    plt.text(6, 1.1e-1, f"Ray ID: {ray_id}\nSI = {SI[ray_id]:0.2f}")

    plt.xlabel("$ I / \\langle I \\rangle$")
    plt.xlim([0, 10])
    plt.xticks(np.arange(0, 10, 3))

    if k % 3 != 0:
        axes[k].set_yticklabels("")
    else:
        axes[k].set_ylabel("probability density")

    if k >= 15:
        plt.xlabel("$ I / \\langle I \\rangle$")
    else:
        plt.xlabel("")
        axes[k].set_xticklabels("")
    # plt.title(ray_id, loc='left', fontsize=10, pad=-3)

    # plt.text(-5, 3e-1, f'SI = {SI[ray_id]:0.2f}')

plt.tight_layout(pad=0)

fig.savefig(
    "../../figures/Boundary_Intensity_Distribution_linear.pdf", bbox_inches="tight"
)

In [ ]:
plt.rcParams.update({"font.size": 8})
plt.rcParams.update({"text.usetex": True})

fig, axes = plt.subplots(6, 3, figsize=(6.68, 7))
axes = axes.flatten()

for k, ray_id in enumerate(boundary_peaks):
    # Intensity Distribution
    plt.sca(axes[k])

    # Log-Intensity Distribution
    plt.sca(axes[k])
    _ = plt.hist(
        lI_norm[ray_id],
        bins=np.linspace(-6, 3, 40),
        alpha=0.3,
        density=True,
        color="gray",
    )

    plt.scatter(
        y=3e-3 * np.random.uniform(0.7, 1.3, 644),
        x=lI_norm[ray_id].flatten(),
        lw=0,
        alpha=0.3,
        s=1,
        c="k",
    )

    x_plot = np.linspace(-6, 4, 1000)
    plt.plot(x_plot, np.exp(x_plot - np.exp(x_plot)), "-k")

    sigma = 1
    f_norm = 1 / (sigma * np.sqrt(2 * np.pi)) * np.exp(-0.5 * (x_plot / sigma) ** 2)
    plt.plot(x_plot, f_norm, "--k")

    plt.yscale("log")
    plt.ylim([1e-3, 1e0])
    plt.xlim([-4, 4])
    plt.xticks(np.arange(-6, 4, 3))
    plt.yticks(np.logspace(-2, 0, 3))
    plt.text(-5.5, 2e-1, f"Ray ID: {ray_id}\nSI = {SI[ray_id]:0.2f}")

    if k % 3 != 0:
        axes[k].set_yticklabels("")
    else:
        axes[k].set_ylabel("probability density")

    if k >= 15:
        plt.xlabel(
            "$(\\mathrm{ln \\ I} - \\langle \\mathrm{ln \\ I} \\rangle) / \\sigma_{\\mathrm{ln \\ I}}$"
        )
    else:
        plt.xlabel("")
        axes[k].set_xticklabels("")
    # plt.title(ray_id, loc='left', fontsize=10, pad=-3)

    # plt.text(-5, 3e-1, f'SI = {SI[ray_id]:0.2f}')

plt.tight_layout(pad=0)


fig.savefig(
    "../../figures/Boundary_Intensity_Distribution_log.pdf", bbox_inches="tight"
)